In [11]:
import torch
import numpy as np 
import os 
from PIL import Image
import pandas as pd 
import json
import ast
import cv2
import h5py


import torch 
from torch.utils.data import Dataset
from torchvision import transforms
import collections


In [12]:

def resize_keypoints(keypoints, original_size, new_size):
    """
    Resize keypoints to match the new image size.

    Args:
        keypoints (list of tuple): List of keypoint coordinates [(x, y), ...]. for One person
        original_size (tuple): Original image size (H, W).
        new_size (tuple): Target image size (H, W).

    Returns:     
        list of tuple: Resized keypoint coordinates [(x', y'), ...]. for one person
    """
    orig_h, orig_w = original_size
    new_h, new_w = new_size
    scale_h, scale_w = new_h / orig_h, new_w / orig_w

    resized_keypoints = [(x * scale_w, y * scale_h) for x, y in keypoints]
    return resized_keypoints


limbs_pairs_person  = [
    (0, 5),   # Nose → Left Shoulder
    (0, 6),   # Nose → Right Shoulder
    (0, 1),   # Nose → Left Eye
    (0, 2),   # Nose → Right Eye
    (1, 3),   # Left Eye → Left Ear
    (2, 4),   # Right Eye → Right Ear
    (5, 6),   # Left Shoulder → Right Shoulder
    (5, 7),   # Left Shoulder → Left Elbow
    (7, 9),   # Left Elbow → Left Wrist
    (6, 8),   # Right Shoulder → Right Elbow
    (8, 10),  # Right Elbow → Right Wrist
    (5, 11),  # Left Shoulder → Left Hip
    (6, 12),  # Right Shoulder → Right Hip
    (11, 12), # Left Hip → Right Hip
    (11, 13), # Left Hip → Left Knee
    (13, 15), # Left Knee → Left Ankle
    (12, 14), # Right Hip → Right Knee
    (14, 16)  # Right Knee → Right Ankle
    ]
    
import numpy as np

def build_limb_vector_for_skeleton(gt_keypoints, bone_mapping):
    """
    Given a list of keypoints and a bone mapping, returns:
      1) limb_tensor:  A single tensor of shape (4 * L), 
         where L is the number of limbs. Each limb contributes [x1, y1, x2, y2].
      2) adjacency:    A (N x N) tensor indicating which joints are connected. 

    Args:
      gt_keypoints (list of tuples): [(x0, y0), (x1, y1), ..., (xN-1, yN-1)]
      bone_mapping (list of tuples): [(j1, j2), (j3, j4), ...]
        Each (j1, j2) indicates a limb between joint j1 and j2.

    Returns:
      dict with:
        - "limb_tensor": torch.Tensor of shape (4*L,)
        - "adjacency":   torch.Tensor of shape (N, N)
    """

    # 1) Concatenate limb endpoints into a single list
    #    [x1, y1, x2, y2, x3, y3, x4, y4, ...]
    limb_coords = []
    for (j1, j2) in bone_mapping:
        y1, x1 = gt_keypoints[j1]
        y2, x2 = gt_keypoints[j2] 
        limb_coords.extend([x1, y1, x2, y2])

    # Convert to a PyTorch tensor (shape: [4*L])
    limb_tensor = torch.tensor(limb_coords, dtype=torch.float32)

    # 2) Build an NxN adjacency matrix as a tensor
    N = len(bone_mapping)
    adjacency = torch.zeros((N, N), dtype=torch.float32)
    for (j1, j2) in bone_mapping:
        adjacency[j1, j2] = 1.0
        adjacency[j2, j1] = 1.0  # if undirected

    return {
        "limb_tensor": limb_tensor,  # shape (4*L,)
        "adjacency":   adjacency     # shape (N, N)
    }




In [13]:
    
    # VGAF transforms
general_transforms = {
        'train': transforms.Compose([
            transforms.RandomResizedCrop(size=(224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]),
        'val': transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
    }

In [14]:
import os
import json
import torch
from PIL import Image
import shutil

def SaveSkeletonsVideo(
    data_folder_src,
    csv_file_src,
    data_folder_dst,
    csv_file_dst,
    save_imgs=False,
    nb_frames=5,
    score=0.1
):
    # create dir to save data
    if os.path.exists(data_folder_dst):
        shutil.rmtree(data_folder_dst, ignore_errors=True)
    os.makedirs(data_folder_dst, exist_ok=True)
    
    with open(os.path.join(csv_file_src), 'r') as file:
        annotations = json.load(file)

    keep_files_names = []

    for index in range(len(annotations)):
        annotation = annotations[index]
        label_emotion = annotation['emotion_label']
        file_name = annotation['file_name']
        video_folder_src = os.path.join(data_folder_src, file_name)

        # Storage for frames, skeletons, adjacency
        Frames = []
        One_video_person_kp = []
        One_video_person_adj = []
        keep_frames_names = []

        # ---- Loop over frames in this video
        for frame_annotation in annotation.get("person_annotations", []):
            frame_name = frame_annotation["frame_name"]
            frame_path = os.path.join(video_folder_src, frame_name)

            # -- Load and process the image
            image_context = Image.open(frame_path)
            if image_context.mode != 'RGB':
                image_context = image_context.convert('RGB')
            original_size = image_context.size

            new_size = (224, 224)
            image_frame = image_context.resize(new_size, Image.LANCZOS)
            image_context = general_transforms['train'](image_frame)  # your transform

            # -- People in this frame
            persons = frame_annotation["persons"][0]
            if not persons['pose']:
                keypoints_list = []
            else:
                # Filter out low confidence keypoints
                keypoints_list = [
                    [
                        kp[:2] if kp[-1] >= score else (-1, -1)
                        for kp in keypoints
                    ]
                    for keypoints in persons['pose'][0]
                ]

            # -- Resize keypoints to new image size
            keypoints_list_resized = []
            if len(keypoints_list) != 0:
                keypoints_list_resized = [
                    resize_keypoints(person_kp, original_size, new_size)
                    for person_kp in keypoints_list
                ]

            # -- Build skeleton and adjacency
            sekeleton_person_gt = []
            adjacency_person_gt = []
            for pers_id in range(len(keypoints_list_resized)):
                skeleton_data = build_limb_vector_for_skeleton(
                    keypoints_list_resized[pers_id],
                    limbs_pairs_person
                )
                sekeleton_person_gt.append(skeleton_data["limb_tensor"])
                adjacency_person_gt.append(skeleton_data["adjacency"])

            # --------------------------------------------------------------
            # Skip frames that have no persons
            # --------------------------------------------------------------
            if len(sekeleton_person_gt) == 0:
                # i.e., no persons => skip this frame
                continue

            # Now we can safely stack them for this frame
            skt_frame_person = torch.stack(sekeleton_person_gt)  # shape: [num_persons, ...]
            adj_frame_person = torch.stack(adjacency_person_gt)  # shape: [num_persons, ...]

            # Append to the list of frames for this video
            One_video_person_kp.append(skt_frame_person)
            One_video_person_adj.append(adj_frame_person)
            Frames.append(image_context)

            frame_info = {
                "frame_name": frame_name,
                "persons": persons,
            }
            keep_frames_names.append(frame_info)

        # --------------------------------------------------------------
        # If after processing all frames, no persons => skip the video
        # --------------------------------------------------------------
        if len(One_video_person_kp) == 0:
            print(f"Skipping video {file_name}: no valid frames with persons.")
            continue

        # --------------------------------------------------------------
        # 1) Find max number of persons across frames in this video
        # --------------------------------------------------------------
        max_num_persons = max(frame_kp.shape[0] for frame_kp in One_video_person_kp)

        # --------------------------------------------------------------
        # 2) Pad each frame's person dimension to max_num_persons
        # --------------------------------------------------------------
        padded_person_kp = []
        padded_person_adj = []
        for i in range(len(One_video_person_kp)):
            current_skp = One_video_person_kp[i]
            current_adj = One_video_person_adj[i]
            current_num_persons = current_skp.shape[0]

            if current_num_persons < max_num_persons:
                last_person_skp = current_skp[-1].unsqueeze(0)
                reps = max_num_persons - current_num_persons
                # shape: [max_num_persons, ...]
                padded_skp = torch.cat(
                    [current_skp, last_person_skp.expand(reps, *last_person_skp.shape[1:])],
                    dim=0
                )

                last_person_adj = current_adj[-1].unsqueeze(0)
                padded_adj = torch.cat(
                    [current_adj, last_person_adj.expand(reps, *last_person_adj.shape[1:])],
                    dim=0
                )
            else:
                padded_skp = current_skp
                padded_adj = current_adj

            padded_person_kp.append(padded_skp)
            padded_person_adj.append(padded_adj)

        One_video_person_kp = padded_person_kp
        One_video_person_adj = padded_person_adj

        # --------------------------------------------------------------
        # 3) Pad the number of frames if needed (temporal dimension)
        # --------------------------------------------------------------
       
        current_nb_frames = len(Frames)
        if current_nb_frames < nb_frames:
            while len(One_video_person_kp) < nb_frames:
                One_video_person_kp.append(One_video_person_kp[-1])
                One_video_person_adj.append(One_video_person_adj[-1])
                Frames.append(Frames[-1])

        # --------------------------------------------------------------
        # 4) Save the data
        # --------------------------------------------------------------
        output_person_kp = os.path.join(data_folder_dst, f"{file_name}_person_kp.pt")
        output_person_adj = os.path.join(data_folder_dst, f"{file_name}_person_adj.pt")
        output_imgs = os.path.join(data_folder_dst, f"{file_name}.pt")

        # shape: [T, max_num_persons, ...]
        torch.save(torch.stack(One_video_person_kp), output_person_kp)
        torch.save(torch.stack(One_video_person_adj), output_person_adj)

        if save_imgs:
            torch.save(torch.stack(Frames), output_imgs)
            print("saved:", output_imgs)

        print("saved:", output_person_kp)

        # Record info for writing JSON
        if len(One_video_person_kp) != 0:
            data_info = {
                "file_name": file_name,
                "label_emotion": label_emotion,
                "frames": keep_frames_names,
            }
            keep_files_names.append(data_info)

        # (Optionally remove the break if you want to process *all* videos)
        #break

    # Finally, write out the new CSV/JSON with the updated data
    with open(csv_file_dst, 'w') as file:
        json.dump(keep_files_names, file, indent=4)


In [ ]:
# csv_file_src =  "...../VGAF_Video_images_5/train_video_annotation_vitpose.json"
# data_folder_src= '....../VGAF_Video_images_5/Train'

# csv_file_dst =  "....../VGAF_Video_images_5/train_video_annotation_all_vitpose_kp.json"
# data_folder_dst= '...../VGAF_Video_images_5/Train_skeleton_tensor'

# SaveSkeletonsVideo(data_folder_src,csv_file_src, data_folder_dst, csv_file_dst,save_imgs=True, score=0.1)